# Backtest 示例

使用字典定义日频行情数据集，通过 `run_backtest` 运行最小 DolphinDB
Backtest 生命周期并按需读取结果。

## Goal

1. 将 CoreData 原始字段映射为 Backtest 所需字段。
2. 定义合法的 DolphinDB 生命周期回调。
3. 读取消息数、组合净值、收益汇总和成交明细。
4. 使用 `with` 确保结果 session 被关闭。

示例回调故意不下单，用于验证数据和生命周期链路；实际策略只需替换 `onBar`。

## Setup

DolphinDB 连接参数从项目的 `.env` 或 `DOLPHIN_HOST`、`DOLPHIN_PORT`、
`DOLPHIN_USERNAME`、`DOLPHIN_PASSWORD` 环境变量读取。实时执行需要
CoreData 表中已有示例数据，并已安装 `MatchingEngineSimulator` 与
`Backtest` 插件。

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
load_dotenv(project_root / ".env")
load_dotenv(project_root.parent / ".env")

from core import run_backtest
from core.utils import logger
logger.remove()
logger.add(sys.stderr, level="INFO")

2

## Steps

### 1. 定义回测数据

Backtest 消息要求 `open`、`low`、`high`、`close`、`volume`、
`upLimitPrice`、`downLimitPrice` 和 `prevClosePrice`。
本示例传入 `adj="qfq"`，使用 `adj_factor` 对这些价格字段前复权。

In [2]:
dataset_query = {
    "start_date": "2025-01-01",
    "end_date": "2025-03-31",
    "lookback": "30D",
    "codes": ["000001.SZ", "600000.SH"],
    "factors": [
        "open", "low", "high", "close", "vol",
        "up_limit", "down_limit", "pre_close",
    ],
    "derivatives": {
        "volume": {
            "type": "DIRECT", "op": "unary.cast",
            "fields": {"col": "vol"},
            "params": {"dtype": "long"},
        },
        "upLimitPrice": {
            "type": "DIRECT", "op": "unary.cast",
            "fields": {"col": "up_limit"},
            "params": {"dtype": "double"},
        },
        "downLimitPrice": {
            "type": "DIRECT", "op": "unary.cast",
            "fields": {"col": "down_limit"},
            "params": {"dtype": "double"},
        },
        "prevClosePrice": {
            "type": "DIRECT", "op": "unary.cast",
            "fields": {"col": "pre_close"},
            "params": {"dtype": "double"},
        },
        "moving_average_20d": {
            "type": "TS", "op": "unary.rolling_mean",
            "fields": {"col": "close"},
            "params": {"window": 20, "min_periods": 20},
        },
        "above_average": {
            "type": "DIRECT", "op": "binary.gt",
            "fields": {"left": "close", "right": "moving_average_20d"},
            "params": {},
        },
    },
    "filters": ["above_average"],
}

print("codes:", dataset_query["codes"])
print("output columns:", [*dataset_query["factors"], *dataset_query["derivatives"]])

codes: ['000001.SZ', '600000.SH']
output columns: ['open', 'low', 'high', 'close', 'vol', 'up_limit', 'down_limit', 'pre_close', 'volume', 'upLimitPrice', 'downLimitPrice', 'prevClosePrice', 'moving_average_20d', 'above_average']


### 2. 定义生命周期回调

回调值是完整的 DolphinDB 函数源码。函数名可以自定义，字典键决定生命周期位置。

In [3]:
utils = {
    "identityExample": '''
        def identityExample(value) {
            return value
        }
    ''',
}
callbacks = {
    "initialize": '''
        def initialize(mutable context) {
            return NULL
        }
    ''',
    "onBar": '''
        def onBar(mutable context, message, indicator) {
            // 最小示例不下单；在这里实现实际策略逻辑。
            return NULL
        }
    ''',
    "finalize": '''
        def finalize(mutable context) {
            return NULL
        }
    ''',
}
print("utils:", list(utils))
print("callbacks:", list(callbacks))

utils: ['identityExample']
callbacks: ['initialize', 'onBar', 'finalize']


### 3. 运行并读取结果

结果数据保存在 `BacktestResult.session` 中。属性只在访问时执行对应 DOS；
退出 `with` 后 session 自动关闭。

In [4]:
backtest_result= run_backtest(
    dataset_query,
    callbacks,
    utils=utils,
    adj="qfq",
    name="runtime_example",
    config={"cash": 1_000_000.0},
)

2026-07-30 00:28:03.803 | INFO     | core.database.session:create_session:45 - DolphinDB: 127.0.0.1:8848


2026-07-30 00:28:04.472 | SUCCESS  | core.apps.backtest.api:run_backtest:209 - 回测完成：name=runtime_example，结果将在访问 BacktestResult 成员时生成，耗时=0.67 秒


In [5]:
daily_portfolios = backtest_result.daily_portfolios
return_summary = backtest_result.return_summary
trade_details = backtest_result.trade_details

print("engine name:", backtest_result.name)
print("session type:", type(backtest_result.session).__name__)
display(return_summary)
display(daily_portfolios.tail(10))
display(trade_details.head(10))

engine name: runtime_example
session type: Session


,totalReturn,annualReturn,annualVolatility,annualSkew,annualKur,sharpeRatio,maxDrawdown,drawdownRatio,beta,alpha,annualExcessReturn,benchmarkReturn,turnoverRate,dailyWinningRate
0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0


,tradeDate,floatingPnl,realizedPnl,totalPnl,cash,totalMarketValue,totalEquity,netValue,totalReturn,ratio,pnl,totalFee,frozenFunds
47,2025-03-18,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
48,2025-03-19,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
49,2025-03-20,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
50,2025-03-21,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
51,2025-03-24,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
52,2025-03-25,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
53,2025-03-26,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
54,2025-03-27,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
55,2025-03-28,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0
56,2025-03-31,0.0,0.0,0.0,1000000.0,0.0,1000000.0,1.0,0.0,0.0,0.0,0.0,0.0


,orderId,symbol,direction,sendTime,orderPrice,orderQty,tradeTime,tradePrice,tradeQty,orderStatus,label


In [6]:
backtest_result.close()

## Checks

In [7]:
required_outputs = {
    "open", "low", "high", "close", "volume",
    "upLimitPrice", "downLimitPrice", "prevClosePrice",
}
actual_outputs = set(dataset_query["factors"]) | set(dataset_query["derivatives"])
assert required_outputs <= actual_outputs
assert set(callbacks) == {"initialize", "onBar", "finalize"}
assert all(isinstance(definition, str) for definition in utils.values())
assert all(isinstance(definition, str) for definition in callbacks.values())
assert backtest_result.closed
assert daily_portfolios is not None
assert return_summary is not None
assert trade_details is not None

print("Backtest example checks passed.")

Backtest example checks passed.


## Next Steps

- 在 `onBar` 中实现实际策略逻辑并提交订单。
- 增加 `beforeTrading`、`onOrder`、`onTrade` 或 `afterTrading` 回调。
- `BacktestResult` 的属性会在访问时从仍存活的回测引擎生成结果。
- 使用 `backtest_result.download("任意 DOS 代码")` 读取同一 session 的其他变量。